In [1]:
import torch

from src.models.convnextv2_backbone import (
    convnextv2_atto_backbone,
    convnextv2_tiny_backbone,
)

/home/mj/.local/share/mamba/envs/faiss-final/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/mj/.local/share/mamba/envs/faiss-final/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")

In [3]:
MODEL: str = "convnextv2_tiny"  # or "convnextv2_tiny"

if MODEL == "convnextv2_tiny":
    convnextv2_model = convnextv2_tiny_backbone(num_classes=200)
    print(f"Model: {MODEL} | Timestamp: {timestamp}")
elif MODEL == "convnextv2_atto":
    convnextv2_model = convnextv2_atto_backbone(num_classes=200)
    print(f"Model: {MODEL} | Timestamp: {timestamp}")

Model: convnextv2_tiny | Timestamp: 2025-05-16 13:10:23


In [4]:
# load weights
if MODEL == "convnextv2_tiny":
    WEIGHT_PATH = "checkpoints/convnextv2_tiny_1k_224_fcmae.pt"
elif MODEL == "convnextv2_atto":
    WEIGHT_PATH = "checkpoints/convnextv2_atto_1k_224_fcmae.pt"

state_dict = torch.load(WEIGHT_PATH, weights_only=True)
print(state_dict["model"].keys())

odict_keys(['downsample_layers.0.0.bias', 'downsample_layers.0.0.weight', 'downsample_layers.0.1.bias', 'downsample_layers.0.1.weight', 'downsample_layers.1.0.bias', 'downsample_layers.1.0.weight', 'downsample_layers.1.1.bias', 'downsample_layers.1.1.weight', 'downsample_layers.2.0.bias', 'downsample_layers.2.0.weight', 'downsample_layers.2.1.bias', 'downsample_layers.2.1.weight', 'downsample_layers.3.0.bias', 'downsample_layers.3.0.weight', 'downsample_layers.3.1.bias', 'downsample_layers.3.1.weight', 'stages.0.0.grn.beta', 'stages.0.0.grn.gamma', 'stages.0.0.dwconv.bias', 'stages.0.0.dwconv.weight', 'stages.0.0.norm.bias', 'stages.0.0.norm.weight', 'stages.0.0.pwconv1.bias', 'stages.0.0.pwconv1.weight', 'stages.0.0.pwconv2.bias', 'stages.0.0.pwconv2.weight', 'stages.0.1.grn.beta', 'stages.0.1.grn.gamma', 'stages.0.1.dwconv.bias', 'stages.0.1.dwconv.weight', 'stages.0.1.norm.bias', 'stages.0.1.norm.weight', 'stages.0.1.pwconv1.bias', 'stages.0.1.pwconv1.weight', 'stages.0.1.pwconv2.bi

In [5]:
convnextv2_model.load_state_dict(state_dict["model"])

<All keys matched successfully>

In [6]:
# freeze the model features
for param in convnextv2_model.parameters():
    param.requires_grad = False

In [7]:
# train the model on CUB dataset

from src.datasets.cub import get_cub200

train_ds, val_ds = get_cub200(
    "/home/mj/source/codebook_playground/codebook_train/data",
    256,
    224,
    0.1,
    0.5,
)

train_dl = torch.utils.data.DataLoader(
    train_ds, batch_size=128, shuffle=True, num_workers=8, pin_memory=True
)
val_dl = torch.utils.data.DataLoader(
    val_ds, batch_size=128, shuffle=False, num_workers=8, pin_memory=True
)

In [9]:
import torch
import torch.nn as nn
import faiss
import numpy as np
from torch.utils.data import DataLoader


@torch.no_grad()
def extract_features(
    model_encoder: nn.Module,
    dataloader: DataLoader,
    device: str = "cuda",
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Extracts features and labels from a dataloader using the given encoder.

    Args:
        model_encoder (nn.Module): The feature encoder model.
        dataloader (DataLoader): DataLoader for the dataset.
        device (str): Device to run the model on ('cuda' or 'cpu').

    Returns:
        tuple[torch.Tensor, torch.Tensor]: A tuple containing:
            - features_tensor: Tensor of extracted features.
            - labels_tensor: Tensor of corresponding labels.
    """

    is_train = model_encoder.training
    if is_train:
        model_encoder.eval()

    features_list = []
    labels_list = []

    for images, labels in dataloader:
        images = images.to(device)
        out = model_encoder(images)
        if isinstance(out, tuple):
            features: torch.Tensor = out[0]
        else:
            features: torch.Tensor = out

        features = features.mean(dim=[2, 3])  # Global average pooling
        features = nn.functional.normalize(features, dim=1)

        features_list.append(features.cpu())
        labels_list.append(labels)

    features_tensor = torch.cat(features_list, dim=0)
    labels_tensor = torch.cat(labels_list, dim=0)

    if is_train:
        model_encoder.train()

    return features_tensor, labels_tensor


def evaluate_knn(
    model_encoder: nn.Module,
    train_dataloader_knn: torch.utils.data.DataLoader,
    val_dataloader_knn: torch.utils.data.DataLoader,
    k: int = 10,
    device: str = "cuda",
):
    print("Extracting training features...")
    train_features, train_labels = extract_features(
        model_encoder, train_dataloader_knn, device
    )
    train_features_np = train_features.cpu().numpy()
    train_labels_np = train_labels.cpu().numpy()

    print("Extracting validation features...")
    val_features, val_labels = extract_features(
        model_encoder, val_dataloader_knn, device
    )
    val_features_np = val_features.cpu().numpy()
    val_labels_np = val_labels.cpu().numpy()

    d = train_features_np.shape[1]  # Dimension of features

    print("Building Faiss index...")
    # Simple flat L2 index (exact search, good for moderate size or GPU)
    index = faiss.IndexFlatL2(d)

    gpu_resource = faiss.StandardGpuResources()
    index = faiss.index_cpu_to_gpu(gpu_resource, 0, index)

    index.add(train_features_np.astype(np.float32))  # Add gallery features to index
    print(f"Faiss index built with {index.ntotal} vectors.")

    print(f"Searching KNN (k={k}) with Faiss...")
    # D: distances, I: indices of neighbors in the gallery
    _, indices = index.search(val_features_np.astype(np.float32), k)
    print(indices)
    predictions = []
    for neighbors_indices in indices:
        neighbor_actual_labels = train_labels_np[neighbors_indices]
        # Simple majority vote
        pred_label = np.bincount(neighbor_actual_labels).argmax()
        predictions.append(pred_label)

    predictions_np = np.array(predictions)
    correct = (predictions_np == val_labels_np).sum()
    accuracy = correct / len(val_labels_np)

    return accuracy * 100

In [10]:
from sklearn.neighbors import KNeighborsClassifier


# --- KNN Evaluation using scikit-learn ---
def evaluate_knn_sklearn(
    model_encoder: nn.Module,
    train_dataloader_knn: DataLoader,
    val_dataloader_knn: DataLoader,
    k: int = 10,
    device: str = "cuda",
) -> float:
    """
    Evaluates k-NN classification accuracy using scikit-learn.

    Args:
        model_encoder (nn.Module): The feature encoder model.
        train_dataloader_knn (DataLoader): DataLoader for the training set (gallery).
        val_dataloader_knn (DataLoader): DataLoader for the validation set (queries).
        k (int): Number of neighbors for k-NN.
        device (str): Device for feature extraction ('cuda' or 'cpu').

    Returns:
        float: k-NN classification accuracy (%).
    """
    print("Extracting training features...")
    train_features, train_labels = extract_features(
        model_encoder, train_dataloader_knn, device
    )
    train_features_np = train_features.cpu().numpy()
    train_labels_np = train_labels.cpu().numpy()

    print("Extracting validation features...")
    val_features, val_labels = extract_features(
        model_encoder, val_dataloader_knn, device
    )
    val_features_np = val_features.cpu().numpy()
    val_labels_np = val_labels.cpu().numpy()

    print(f"Building and evaluating scikit-learn KNN (k={k})...")
    knn_classifier = KNeighborsClassifier(
        n_neighbors=k,
        n_jobs=4,
        p=2,
    )
    knn_classifier.fit(train_features_np, train_labels_np)
    accuracy = knn_classifier.score(val_features_np, val_labels_np)

    return accuracy * 100

In [11]:
def train_epoch_codebook_ssl(
    model: torch.nn.Module,
    train_dl: torch.utils.data.DataLoader,
    optimizer: torch.optim.Optimizer,
):
    """Train the model through codebook loss assuming self-supervised learning.
    Evaluate using KNN.

    Args:
        model (torch.nn.Module): The model to train.
        train_dl (torch.utils.data.DataLoader): The dataloader for training.
        optimizer (torch.optim.Optimizer): The optimizer to use.
    """

    model.train()
    for i, (x, y) in enumerate(train_dl):
        x = x.cuda()
        y = y.cuda()

        optimizer.zero_grad()
        _, codebook_loss = model(x)
        codebook_loss.backward()
        optimizer.step()

        if i % (len(train_dl) // 10) == 0:
            print(f"Iter {i}/{len(train_dl)} Loss {codebook_loss.item()}")

In [12]:
from torch import nn


class CNNCodebookWrapper(nn.Module):
    def __init__(self, features: nn.Module, codebook: nn.Module, classifier: nn.Module):
        """
        Wrapper for a CNN model with a codebook and classifier.
        This wrapper allows for separation of codebook and classifier loss calculations.

        Args:
            features (nn.Module): Feature extractor module.
            codebook (nn.Module): Codebook module.
            classifier (nn.Module): Classifier module.
        """

        super().__init__()
        self.features = features
        self.codebook = codebook
        self.classifier = classifier

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        x = self.features(x)
        x, codebook_loss = self.codebook(x)
        x = self.classifier(x)

        return x, codebook_loss

In [13]:
from src.codebook import CosineSimilarityCodebook

codebook = CosineSimilarityCodebook(
    num_entries=1024, embedding_dim=768, mapping_dim_config=[], entropy_loss_weight=0.0
)

codebook_wrapper = CNNCodebookWrapper(
    features=convnextv2_model,
    codebook=codebook,
    classifier=nn.Identity(),  # No classifier
)

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [15]:
convnextv2_model.to(device)
codebook_wrapper.to(device)

CNNCodebookWrapper(
  (features): ConvNeXtV2Backbone(
    (downsample_layers): ModuleList(
      (0): Sequential(
        (0): Conv2d(3, 96, kernel_size=(4, 4), stride=(4, 4))
        (1): LayerNorm()
      )
      (1): Sequential(
        (0): LayerNorm()
        (1): Conv2d(96, 192, kernel_size=(2, 2), stride=(2, 2))
      )
      (2): Sequential(
        (0): LayerNorm()
        (1): Conv2d(192, 384, kernel_size=(2, 2), stride=(2, 2))
      )
      (3): Sequential(
        (0): LayerNorm()
        (1): Conv2d(384, 768, kernel_size=(2, 2), stride=(2, 2))
      )
    )
    (stages): ModuleList(
      (0): Sequential(
        (0): Block(
          (dwconv): Conv2d(96, 96, kernel_size=(7, 7), stride=(1, 1), padding=(3, 3), groups=96)
          (norm): LayerNorm()
          (pwconv1): Linear(in_features=96, out_features=384, bias=True)
          (act): GELU(approximate='none')
          (grn): GRN()
          (pwconv2): Linear(in_features=384, out_features=96, bias=True)
          (drop_

In [16]:
faiss_knn_acc = evaluate_knn(
    convnextv2_model.to(device),
    train_dl,
    val_dl,
    k=10,
    device=device,
)
print(f"Initial Faiss KNN accuracy: {faiss_knn_acc:.2f}%")

Extracting training features...
Extracting validation features...
Building Faiss index...
Faiss index built with 5994 vectors.
Searching KNN (k=10) with Faiss...
[[5078  160 1557 ... 1450 5347 5593]
 [ 888 3926 1295 ... 3915 5831 2105]
 [2113 4285 4490 ... 2540 3975  706]
 ...
 [2057 2748  758 ... 1715 4411  227]
 [4795 1779 1620 ... 5122 1837 1858]
 [3545 1121 5094 ... 4181  215 5919]]
Initial Faiss KNN accuracy: 3.00%


In [ ]:
optimizer = torch.optim.Adam(codebook.parameters(), lr=0.005)
for epoch in range(5):
    print(f"Epoch {epoch}")
    train_epoch_codebook_ssl(codebook_wrapper, train_dl, optimizer)
    knn_acc = evaluate_knn(
        codebook_wrapper,
        train_dl,
        val_dl,
        k=10,
        device=device,
    )
    print(f"Epoch {epoch} KNN accuracy: {knn_acc:.2f}%")

Epoch 0
Iter 0/47 Loss 0.2199702113866806
Iter 4/47 Loss 0.20594888925552368
Iter 8/47 Loss 0.20666168630123138
Iter 12/47 Loss 0.20019415020942688
Iter 16/47 Loss 0.19440940022468567
Iter 20/47 Loss 0.19328078627586365
Iter 24/47 Loss 0.1986033171415329
Iter 28/47 Loss 0.1802135854959488
Iter 32/47 Loss 0.17526111006736755
Iter 36/47 Loss 0.17738336324691772
Iter 40/47 Loss 0.15621346235275269
Iter 44/47 Loss 0.1668616086244583
Extracting training features...
Extracting validation features...
